# 📈 Identificación de Regímenes de Mercado Financiero mediante Modelos Ocultos de Markov (HMM)

**Autores:**
- Sebastian Garces Cadavid (`sebastian.garces@est.iudigital.edu.co`)
- Sofia Miranda Cubides (`sofia.miranda@est.iudigital.edu.co`)
- Oscar Rene Peñaloza Ardila (`oscar.penaloza@est.iudigital.edu.co`)
- Andres Mauricio Quintero Rios (`andres.quintero@est.iudigital.edu.co`)

**Docente Asesor:** Dr. Julián Andrés Loaiza López (`julian.loaiza@prof.iudigital.edu.co`)  
**Institución:** Institución Universitaria Digital de Antioquia (IU Digital)  
**Curso:** Simulación — Noviembre de 2026

---
### 🎯 Objetivos del Cuaderno:
1. Implementar la clase `HMMFinanciero` formalizada con $\lambda = (S, V, A, B, \pi)$.
2. Simular $T=50$ períodos de trading con generación geométrica de precios.
3. Decodificar la trayectoria óptima con el algoritmo de Viterbi en log-probabilidades.
4. Comparar el desempeño frente a una línea base (*baseline*) heurística de ventana deslizante ($k=2$).
5. Visualizar la serie de precios con fondos de color por régimen de mercado (verde/ámbar/rojo).
6. Computar métricas prudenciales de riesgo financiero (VaR 95%, CVaR, Drawdown, Sharpe) y emitir recomendaciones de trading.

In [ ]:
# Instalación de dependencias (si se ejecuta en Google Colab)
!pip install numpy pandas matplotlib scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ Entorno inicializado exitosamente.")

## 1. Definición Formal de la Clase HMM para Mercados Financieros

In [ ]:
class HMMFinanciero:
    def __init__(self, A=None, B=None, pi=None, T=50):
        self.estados = ['Mercado Alcista', 'Mercado Lateral', 'Mercado Bajista']
        self.observaciones = ['Retorno Positivo', 'Retorno Neutral', 'Retorno Negativo']
        self.N = len(self.estados)
        self.M = len(self.observaciones)
        self.T = T
        
        # Colores por régimen
        self.colores = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
        
        # Matriz de transición A
        self.A = np.array(A if A is not None else [
            [0.60, 0.30, 0.10],
            [0.30, 0.40, 0.30],
            [0.10, 0.30, 0.60]
        ], dtype=np.float64)
        
        # Matriz de emisión B
        self.B = np.array(B if B is not None else [
            [0.70, 0.20, 0.10],
            [0.30, 0.40, 0.30],
            [0.10, 0.20, 0.70]
        ], dtype=np.float64)
        
        # Distribución inicial π
        self.pi = np.array(pi if pi is not None else [0.40, 0.30, 0.30], dtype=np.float64)

    def simular(self, T=None, seed=None):
        if T is None: T = self.T
        if seed is not None: np.random.seed(seed)
        
        estados = np.zeros(T, dtype=int)
        obs = np.zeros(T, dtype=int)
        
        estados[0] = np.random.choice(self.N, p=self.pi)
        obs[0] = np.random.choice(self.M, p=self.B[estados[0]])
        
        for t in range(1, T):
            estados[t] = np.random.choice(self.N, p=self.A[estados[t-1]])
            obs[t] = np.random.choice(self.M, p=self.B[estados[t]])
            
        return estados, obs

    def generar_precios(self, estados, obs, p0=100.0):
        T = len(estados)
        params = {0: (0.020, 0.010), 1: (0.000, 0.008), 2: (-0.020, 0.015)}
        mod_o = {0: 0.004, 1: 0.0, 2: -0.004}
        
        precios = np.zeros(T + 1)
        precios[0] = p0
        retornos = np.zeros(T)
        
        for t in range(T):
            mu, sig = params[estados[t]]
            ret = np.random.normal(mu + mod_o[obs[t]], sig)
            retornos[t] = ret
            precios[t+1] = precios[t] * (1.0 + ret)
            
        return precios, retornos

    def viterbi(self, obs):
        T = len(obs)
        delta = np.zeros((T, self.N))
        psi = np.zeros((T, self.N), dtype=int)
        eps = 1e-300
        
        delta[0] = np.log(self.pi + eps) + np.log(self.B[:, obs[0]] + eps)
        
        for t in range(1, T):
            for j in range(self.N):
                trans = delta[t-1] + np.log(self.A[:, j] + eps)
                psi[t, j] = np.argmax(trans)
                delta[t, j] = trans[psi[t, j]] + np.log(self.B[j, obs[t]] + eps)
                
        path = np.zeros(T, dtype=int)
        path[T-1] = np.argmax(delta[T-1])
        for t in range(T-2, -1, -1):
            path[t] = psi[t+1, path[t+1]]
            
        return path, delta[T-1, path[T-1]]

    def baseline_heuristico(self, obs, k=2):
        T = len(obs)
        pred = np.zeros(T, dtype=int)
        for t in range(T):
            sub = obs[max(0, t-k+1):t+1]
            if np.sum(sub == 0) > len(sub)/2: pred[t] = 0
            elif np.sum(sub == 2) > len(sub)/2: pred[t] = 2
            else: pred[t] = 1
        return pred

    def analisis_riesgo(self, estados, retornos, precios):
        vol = np.std(retornos)
        picos = np.maximum.accumulate(precios)
        dd = (precios - picos) / picos
        return {
            'retorno_total': (precios[-1] / precios[0]) - 1.0,
            'volatilidad': vol,
            'var_95': np.percentile(retornos, 5),
            'var_99': np.percentile(retornos, 1),
            'max_drawdown': np.min(dd),
            'sharpe': (np.mean(retornos) / vol) if vol > 0 else 0.0
        }

    def recomendaciones(self, st, r):
        directrices = {
            0: ('🟢 Mercado Alcista', 'COMPRA (Long)', 'Alta (80-100%)', 'Trailing stop y optimización beta.'),
            1: ('🟡 Mercado Lateral', 'NEUTRAL / Rango', 'Moderada (40-50%)', 'Reversión a la media y captura de spreads.'),
            2: ('🔴 Mercado Bajista', 'VENTA / Cobertura', 'Baja (0-15%) o Cash', 'Preservación de capital y activos refugio.')
        }
        nombre, pos, exp, est = directrices[st]
        return {
            'regimen': nombre, 'posicion': pos, 'exposicion': exp, 'estrategia': est,
            'sharpe': f"{r['sharpe']:.3f}", 'max_dd': f"{r['max_drawdown']:.2%}"
        }

print("✅ Clase HMMFinanciero definida correctamente.")

## 2. Simulación de 50 Períodos de Trading y Decodificación de Viterbi

In [ ]:
hmm = HMMFinanciero(T=50)
estados_reales, obs = hmm.simular(seed=42)
precios, retornos = hmm.generar_precios(estados_reales, obs)
viterbi_pred, log_lik = hmm.viterbi(obs)
heur_pred = hmm.baseline_heuristico(obs, k=2)

prec_viterbi = np.mean(estados_reales == viterbi_pred)
prec_heur = np.mean(estados_reales == heur_pred)
riesgo = hmm.analisis_riesgo(estados_reales, retornos, precios)
recs = hmm.recomendaciones(estados_reales[-1], riesgo)

print(f"📊 Precisión Viterbi Global: {prec_viterbi:.2%}")
print(f"📌 Precisión Heurística Baseline (k=2): {prec_heur:.2%}")
print(f"📈 Retorno Total: {riesgo['retorno_total']:.2%}")
print(f"⚠️ Value at Risk (VaR 95%): {riesgo['var_95']:.2%}")
print(f"📉 Máximo Drawdown: {riesgo['max_drawdown']:.2%}")
print(f"💡 Recomendación actual: {recs['regimen']} -> Posición: {recs['posicion']} (Exposición: {recs['exposicion']})")

## 3. Visualización con Matplotlib: Precio Simulado y Fondo Coloreado por Régimen

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(range(len(precios)), precios, color='#111111', linewidth=2.0, label='Precio Simulado S(t)')

for t in range(len(estados_reales)):
    plt.axvspan(t, t + 1, color=hmm.colores[estados_reales[t]], alpha=0.25)

p_alc = mpatches.Patch(color='#2ecc71', alpha=0.6, label='Mercado Alcista')
p_lat = mpatches.Patch(color='#f39c12', alpha=0.6, label='Mercado Lateral')
p_baj = mpatches.Patch(color='#e74c3c', alpha=0.6, label='Mercado Bajista')

plt.title('Simulación de Mercado Financiero mediante HMM (T=50 períodos)', fontsize=13, fontweight='bold')
plt.xlabel('Período de Trading (t)', fontsize=11)
plt.ylabel('Nivel de Precio ($)', fontsize=11)
plt.legend(handles=[p_alc, p_lat, p_baj], loc='upper left', framealpha=0.9)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()